# 🏛️ VoxLegal AI — Phase 7: Fine-Tuning LegalBERT

## Research Task
Fine-tune **LegalBERT** (`nlpaueb/legal-bert-base-uncased`) on the **LEDGAR** dataset for **legal clause type classification**.

## Why This Matters for the Research Paper
- **LegalBERT** was pre-trained on legal corpora (contracts, legislation, court opinions). It understands legal language better than standard BERT.
- **LEDGAR** contains ~60,000 real contract provisions labelled with 100 clause types (Termination, Confidentiality, Payment, Indemnification, etc.).
- Fine-tuning lets the model classify retrieved clauses, adding **explainability** and enabling comparison with baseline models.
- This directly answers **Research Question RQ2**: *Does legal-domain fine-tuning improve legal clause understanding?*

## What We Build
```
LEDGAR Dataset
      ↓
Data Cleaning + Train/Val/Test Split
      ↓
Fine-tune LegalBERT (Clause Classification)
      ↓
Evaluate: Accuracy, Precision, Recall, F1
      ↓
Save Model to HuggingFace Hub
      ↓
Use in VoxLegal AI RAG Pipeline
```

## Runtime: Google Colab T4 GPU (~30-60 min)

> ⚠️ **Go to Runtime → Change runtime type → T4 GPU before running!**

---
## STEP 0 — Verify GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU Memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

---
## STEP 1 — Install Required Libraries

In [ ]:
# Install only what we need
!pip install -q transformers datasets evaluate scikit-learn seaborn huggingface_hub
print('All libraries installed!')

---
## STEP 2 — Load the LEDGAR Dataset

### What is LEDGAR?
- **Source**: Atticus Project (Stanford Law)
- **Contents**: ~60,000 contract provisions from SEC EDGAR filings
- **Labels**: 100 legal clause types (Termination, Confidentiality, Indemnification, etc.)
- **License**: CC BY 4.0 (free for research)
- **Language**: English
- **Why chosen**: Real-world legal contracts, professionally labeled, standard benchmark

In [ ]:
from datasets import load_dataset
import pandas as pd

print('Loading LEDGAR dataset from HuggingFace...')
dataset = load_dataset('lex_glue', 'ledgar')
print('Dataset loaded!')
print(dataset)

# Inspect a sample
print('\n--- Sample provision ---')
sample = dataset['train'][0]
print('Text:', sample['text'][:300])
print('Label (int):', sample['label'])

In [ ]:
# Show label distribution
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Get label names
label_names = dataset['train'].features['label'].names
num_labels = len(label_names)
print(f'Total clause types (labels): {num_labels}')
print('First 20 clause types:', label_names[:20])

# Distribution
label_counts = Counter(dataset['train']['label'])
top_labels = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:20]

plt.figure(figsize=(14, 5))
names = [label_names[l] for l, _ in top_labels]
counts = [c for _, c in top_labels]
plt.barh(names, counts, color='steelblue')
plt.xlabel('Count')
plt.title('Top 20 Clause Types in LEDGAR Training Set')
plt.tight_layout()
plt.show()

print(f'\nTotal training samples: {len(dataset["train"])}')
print(f'Total validation samples: {len(dataset["validation"])}')
print(f'Total test samples: {len(dataset["test"])}')

---
## STEP 3 — Tokenization

### What is tokenization?
Transformers like LegalBERT don't read raw text.
They read **tokens** — pieces of words mapped to numbers.

Example:
```
'Either party may terminate' → [101, 2432, 2283, 2064, 16072, 102]
```

The **tokenizer** converts text → numbers that LegalBERT can process.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 128   # Legal clauses are typically short; 128 tokens is sufficient

print(f'Loading tokenizer from: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded!')

# Test on a sample
sample_text = 'Either party may terminate this agreement by providing 30 days written notice.'
tokens = tokenizer(sample_text, truncation=True, max_length=MAX_LENGTH)
print(f'\nSample text: "{sample_text}"')
print(f'Token IDs: {tokens["input_ids"]}')
print(f'Number of tokens: {len(tokens["input_ids"])}')

In [ ]:
def tokenize_batch(examples):
    """Tokenize a batch of legal provisions."""
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

print('Tokenizing all splits...')
tokenized = dataset.map(tokenize_batch, batched=True, batch_size=256)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print('Tokenization complete!')
print(tokenized)

---
## STEP 4 — Load LegalBERT for Classification

### What are we doing?
We take the pre-trained LegalBERT model and add a **classification head** on top.

```
Input Clause Text
      ↓
LegalBERT Encoder (pre-trained on legal text)
      ↓
[CLS] Token Representation
      ↓
Classification Head (linear layer)
      ↓
Clause Type Label (e.g., "Termination")
```

The encoder weights are **already good at understanding legal language**.
Fine-tuning teaches the new classification head to predict the clause type.

In [ ]:
from transformers import AutoModelForSequenceClassification

print(f'Loading LegalBERT for {num_labels}-class clause classification...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)}
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model loaded on: {device}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

---
## STEP 5 — Define Evaluation Metrics

For the research paper we report:
- **Accuracy** — % of clauses correctly classified
- **Macro F1** — Average F1 across all 100 clause types (treats all classes equally)
- **Micro F1** — Overall F1 weighted by frequency
- **Precision / Recall** — For imbalanced class analysis

In [ ]:
import evaluate
import numpy as np

metric_acc = evaluate.load('accuracy')
metric_f1  = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)['accuracy']
    macro_f1 = metric_f1.compute(predictions=predictions, references=labels, average='macro')['f1']
    micro_f1 = metric_f1.compute(predictions=predictions, references=labels, average='micro')['f1']
    return {
        'accuracy': round(acc, 4),
        'macro_f1': round(macro_f1, 4),
        'micro_f1': round(micro_f1, 4),
    }

print('Evaluation metrics defined: Accuracy, Macro-F1, Micro-F1')

---
## STEP 6 — Training Configuration

### Hyperparameter Choices (explained)
| Parameter | Value | Reason |
|---|---|---|
| learning_rate | 2e-5 | Standard for BERT fine-tuning |
| epochs | 4 | Enough for convergence, not too slow on Colab |
| batch_size | 32 | Fits T4 GPU memory |
| weight_decay | 0.01 | L2 regularization to prevent overfitting |
| warmup_ratio | 0.1 | Warm up learning rate for first 10% of steps |

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

OUTPUT_DIR = '/content/voxlegal_legalbert_ledgar'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_dir=f'{OUTPUT_DIR}/logs',
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # Half-precision on GPU for speed
    report_to='none',                # Disable W&B/MLflow for simplicity
    seed=42,                         # For research reproducibility
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print('Trainer configured!')
print(f'Training samples: {len(tokenized["train"])}')
print(f'Validation samples: {len(tokenized["validation"])}')
print(f'Steps per epoch: {len(tokenized["train"]) // 32}')

---
## STEP 7 — Fine-Tuning!

This is the core training loop.
LegalBERT's weights are updated to classify legal clause types.

**Expected time on T4 GPU: ~30-45 minutes**

In [ ]:
import time

print('Starting fine-tuning...')
print('Model: nlpaueb/legal-bert-base-uncased')
print('Dataset: LEDGAR (LexGLUE benchmark)')
print('Task: Legal Clause Type Classification (100 classes)')
print('Device:', device)
print('='*60)

start = time.time()
trainer.train()
elapsed = time.time() - start

print('='*60)
print(f'Training complete in {elapsed/60:.1f} minutes!')

---
## STEP 8 — Evaluate on Test Set

We evaluate on the **held-out test set** (data the model never saw during training).

These are the numbers you will report in the research paper.

In [ ]:
print('Evaluating on test set...')
test_results = trainer.evaluate(tokenized['test'])

print('\n' + '='*50)
print('TEST SET RESULTS (for research paper)')
print('='*50)
print(f"Accuracy : {test_results['eval_accuracy']:.4f}")
print(f"Macro F1 : {test_results['eval_macro_f1']:.4f}")
print(f"Micro F1 : {test_results['eval_micro_f1']:.4f}")
print('='*50)

In [ ]:
# Per-class analysis — which clause types does the model handle best/worst?
from sklearn.metrics import classification_report
import numpy as np

# Get predictions on test set
preds_output = trainer.predict(tokenized['test'])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

report = classification_report(
    y_true, y_pred,
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

# Show top 15 best and worst performing clause types
report_df = pd.DataFrame(report).T
report_df = report_df[report_df.index.isin(label_names)].sort_values('f1-score', ascending=False)

print('TOP 10 BEST CLASSIFIED CLAUSE TYPES:')
print(report_df[['precision', 'recall', 'f1-score', 'support']].head(10).to_string())
print('\nTOP 10 WORST CLASSIFIED CLAUSE TYPES:')
print(report_df[['precision', 'recall', 'f1-score', 'support']].tail(10).to_string())

In [ ]:
# Confusion matrix heatmap for top 20 most common clause types
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Filter top 20 labels for readability
top20_idx = sorted(Counter(y_true.tolist()).items(), key=lambda x: x[1], reverse=True)[:20]
top20_labels = [i for i, _ in top20_idx]
top20_names  = [label_names[i] for i in top20_labels]

mask = np.isin(y_true, top20_labels)
cm = confusion_matrix(y_true[mask], y_pred[mask], labels=top20_labels)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, xticklabels=top20_names, yticklabels=top20_names, 
            fmt='d', cmap='Blues', annot=True, linewidths=0.5)
plt.title('Confusion Matrix — Top 20 Clause Types (Test Set)', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved to /content/confusion_matrix.png')

---
## STEP 9 — Save Model & Upload to HuggingFace Hub

We save the model so we can:
1. Download it back to the VoxLegal AI backend
2. Reference it in the research paper
3. Allow others to reproduce our results

**You need a free HuggingFace account.** Get a token at: https://huggingface.co/settings/tokens

In [ ]:
# Save locally first
LOCAL_SAVE = '/content/voxlegal_legalbert_final'
trainer.save_model(LOCAL_SAVE)
tokenizer.save_pretrained(LOCAL_SAVE)
print(f'Model saved locally to: {LOCAL_SAVE}')

# Also save test results for the paper
import json
results_to_save = {
    'model': 'nlpaueb/legal-bert-base-uncased',
    'dataset': 'lex_glue/ledgar',
    'task': 'legal_clause_classification',
    'num_labels': num_labels,
    'test_accuracy': test_results['eval_accuracy'],
    'test_macro_f1': test_results['eval_macro_f1'],
    'test_micro_f1': test_results['eval_micro_f1'],
    'hyperparameters': {
        'learning_rate': 2e-5,
        'epochs': 4,
        'batch_size': 32,
        'max_length': MAX_LENGTH,
        'weight_decay': 0.01,
        'seed': 42
    }
}
with open('/content/voxlegal_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)
print('Results saved to /content/voxlegal_results.json')
print(json.dumps(results_to_save, indent=2))

In [ ]:
# OPTIONAL: Upload to HuggingFace Hub
# Replace 'YOUR_HF_USERNAME' with your actual HuggingFace username
# Get your token at: https://huggingface.co/settings/tokens

from huggingface_hub import notebook_login

# This will prompt you to paste your HuggingFace token
notebook_login()

In [ ]:
# Push model to your HuggingFace Hub repo
# Change 'YOUR_HF_USERNAME' to your HuggingFace username
HF_USERNAME = 'YOUR_HF_USERNAME'
REPO_NAME = f'{HF_USERNAME}/voxlegal-legalbert-ledgar'

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)
print(f'Model uploaded to: https://huggingface.co/{REPO_NAME}')
print('Copy this URL — you will need it to download the model into VoxLegal AI backend!')

---
## STEP 10 — Download Files to Your Computer

Download these files from Colab for your project:

In [ ]:
from google.colab import files
import shutil

# Download results JSON (for research paper)
files.download('/content/voxlegal_results.json')

# Download confusion matrix plot (for research paper)
files.download('/content/confusion_matrix.png')

# Zip and download the full model
shutil.make_archive('/content/voxlegal_model', 'zip', LOCAL_SAVE)
files.download('/content/voxlegal_model.zip')

print('Downloads started!')
print('Save these files in your project at: C:/D_drive/voxlegal-ai/research/')

---
## STEP 11 — Test Inference (How the Model Works)

Demonstrate the model classifying a real legal clause.
This is what happens inside VoxLegal AI after integration.

In [ ]:
from transformers import pipeline
import torch

# Load the fine-tuned model for inference
classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    top_k=3
)

# Test on real legal clauses
test_clauses = [
    'Either party may terminate this agreement by providing 30 days written notice to the other party.',
    'The receiving party shall keep all confidential information strictly confidential and shall not disclose it to any third party.',
    'The Company shall pay the Employee a base salary of $85,000 per annum, payable in bi-weekly installments.',
    'Each party shall indemnify, defend, and hold harmless the other party from any claims arising from its breach of this Agreement.',
    'This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware.',
]

print('CLAUSE CLASSIFICATION DEMO')
print('='*70)
for clause in test_clauses:
    result = classifier(clause[:512])
    top = result[0]
    print(f'Clause: "{clause[:80]}..."')
    print(f'  → Predicted Type : {top[0]["label"]} (confidence: {top[0]["score"]:.1%})')
    print(f'  → 2nd best       : {top[1]["label"]} ({top[1]["score"]:.1%})')
    print()

---
## Summary

### What was accomplished:
| Item | Detail |
|---|---|
| Model | LegalBERT (nlpaueb/legal-bert-base-uncased) |
| Dataset | LEDGAR (LexGLUE) — ~60K contract provisions |
| Task | Legal Clause Type Classification (100 classes) |
| Metrics | Accuracy, Macro-F1, Micro-F1 |
| Output | Fine-tuned model + results JSON + confusion matrix |

### Next Steps in VoxLegal AI:
1. Download the fine-tuned model to your machine
2. Use the model to **classify retrieved clauses** in the RAG pipeline
3. Show clause type labels in the Evidence Cards in the frontend
4. Use the results JSON in your research paper (Phase 10)

### Research Paper Note:
```
Document this fine-tuning as:
- Model: nlpaueb/legal-bert-base-uncased
- Dataset: LEDGAR (LexGLUE benchmark, CC BY 4.0)
- Task: Multi-class legal clause classification (100 categories)
- Evaluation: Accuracy, Macro-F1, Micro-F1 on held-out test set
- Seed: 42 (for reproducibility)
```